# Dermatology arm, step 2: train 24 models, then (and only then) predict the external sets

Follows `data/derm/derm_lock_v1.json`.

1. Cleaning and lesion-disjoint 80/10/10 splits for HAM10000 and BCN20000 (seed 20261601).
2. Train 6 architectures x 2 sources x 2 seeds = 24 models (DR full recipes, 8 classes, 15 epochs). Resumable.
3. **Gate**: all 24 checkpoints must exist; their hashes are written to `inference_record.json` before any external prediction is made.
4. Predictions on internal tests (both sources), HIBA and PAD-UFES-20.
5. Fingerprints (H5): perturbation responses on a fixed random half of HIBA patients, unlabelled; shared misses will be scored on the other half.

A100, High-RAM. About 4.5 to 6 hours in total; rerun all cells after any disconnect (finished models are skipped automatically).

Revision 2: evaluation passes fall back to fp32 for any batch whose fp16 output is non-finite (EfficientNet-B3 overflowed at evaluation in revision 1). Training is unchanged.

In [2]:
# Cell 1: setup
from google.colab import drive
drive.mount('/content/drive')
!pip -q install timm
import os, io, json, time, math, random, glob, shutil, hashlib, datetime
import numpy as np, pandas as pd, torch, torch.nn as nn, timm
from concurrent.futures import ThreadPoolExecutor
DERM = '/content/drive/MyDrive/stochastic_orthogonality/data/derm'
CK, PR, FP = f'{DERM}/checkpoints', f'{DERM}/predictions', f'{DERM}/fingerprints'
for d in (CK, PR, FP): os.makedirs(d, exist_ok=True)
LOCK = json.load(open(f'{DERM}/derm_lock_v1.json')); print('lock:', LOCK['lock'], '|', LOCK['written'][:10])
dev = 'cuda'; NW = min(8, os.cpu_count())
torch.backends.cudnn.benchmark = True; torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
def sha(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
CLASSES = ['MEL', 'BCC', 'SCC', 'AK', 'NV', 'BKL', 'DF', 'VASC']; CI = {c: i for i, c in enumerate(CLASSES)}; MAL = [CI['MEL'], CI['BCC'], CI['SCC']]
print(torch.cuda.get_device_name(0))

Mounted at /content/drive
lock: dermatology arm, lock v1 | 2026-09-26
NVIDIA A100-SXM4-40GB


In [3]:
# Cell 1b: rebuild manifests if the intake's final write did not reach Drive (no image downloads; metadata only)
import requests, re, zipfile, time
EXPECTED = {'HAM10000': 11720, 'BCN20000': 18946, 'HIBA': 1616, 'PAD': 2298}      # from the intake printout
if not all(os.path.exists(f'{DERM}/manifest_{s}.csv') for s in EXPECTED):
    API = 'https://api.isic-archive.com/api/v2'; COLL = {'HAM10000': 212, 'BCN20000': 249, 'HIBA': 251}
    S = requests.Session()
    def get_json(url, params=None):
        for i in range(5):
            try: r = S.get(url, params=params, timeout=60); r.raise_for_status(); return r.json()
            except Exception:
                if i == 4: raise
                time.sleep(2 * (i + 1))
    def flatten(d, prefix=''):
        out = {}
        for k, v in (d or {}).items():
            if isinstance(v, dict): out.update(flatten(v, f'{prefix}{k}.'))
            else: out[f'{prefix}{k}'] = v
        return out
    RULES = [('MEL', r'melanoma'), ('BCC', r'basal cell'), ('SCC', r'squamous cell|bowen|in situ, squamous|keratoacanthoma'), ('AK', r'actinic keratosis|akiec'),
             ('NV', r'nev(us|i)'), ('BKL', r'seborrh|lentigo|lichen|keratosis-like|benign keratosis'), ('DF', r'dermatofibroma'), ('VASC', r'angioma|angiokeratoma|pyogenic|hemorrhage|vascular')]
    def map_dx(t):
        t = str(t).lower()
        for c, p in RULES:
            if re.search(p, t): return c
        return None
    MAN = {}
    for key, cid in COLL.items():
        rows, url, params = [], f'{API}/images/search/', {'collections': cid, 'limit': 100}
        while url:
            j = get_json(url, params); params = None
            rows += [{'isic_id': r['isic_id'], 'license': r.get('copyright_license'), **flatten(r.get('metadata'), 'm.')} for r in j['results']]
            url = j.get('next')
        Mx = pd.DataFrame(rows); dxcols = [c for c in Mx.columns if re.search('diagnosis', c, re.I)]
        text = Mx[dxcols].astype(str).agg(' | '.join, axis=1)
        lcol = next((c for c in Mx.columns if c.endswith('lesion_id')), None); pcol = next((c for c in Mx.columns if c.endswith('patient_id')), None)
        MAN[key] = pd.DataFrame({'image_id': Mx.isic_id, 'source': key, 'modality': 'dermoscopy', 'dx_text': text, 'dx': text.map(map_dx),
                                 'lesion_id': Mx[lcol] if lcol else None, 'patient_id': Mx[pcol] if pcol else None, 'license': Mx.license})
        print(key, len(MAN[key]), 'metadata records')
    zp = f'{DERM}/manual/PAD-UFES-20.zip'; os.makedirs('/content/padmeta', exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        names = [n for n in z.namelist() if n.endswith('metadata.csv')]
        if names: z.extract(names[0], '/content/padmeta')
        else: z.extractall('/content/padmeta')
    PADm = pd.read_csv(glob.glob('/content/padmeta/**/metadata.csv', recursive=True)[0])
    PADMAP = {'MEL': 'MEL', 'BCC': 'BCC', 'SCC': 'SCC', 'BOD': 'SCC', 'ACK': 'AK', 'NEV': 'NV', 'SEK': 'BKL'}
    MAN['PAD'] = pd.DataFrame({'image_id': PADm.img_id.str.replace('.png', '', regex=False), 'source': 'PAD', 'modality': 'smartphone', 'dx_text': PADm.diagnostic,
                               'dx': PADm.diagnostic.map(PADMAP), 'lesion_id': PADm.lesion_id.astype(str), 'patient_id': PADm.patient_id.astype(str), 'license': 'CC BY 4.0'})
    for key, df in MAN.items():
        df['malignant'] = df.dx.isin(['MEL', 'BCC', 'SCC']); df['melanoma'] = df.dx.eq('MEL')
        df['path_512'] = [f'{DERM}/images_512/{key}/{i}.jpg' for i in df.image_id]
        present = df.path_512.map(os.path.exists); assert present.all(), f'{key}: {int((~present).sum())} processed images missing'
        with ThreadPoolExecutor(16) as ex: df['raw_sha256'] = list(ex.map(sha, df.path_512))    # hash of the processed image (identical raw bytes give identical output)
        assert len(df) == EXPECTED[key], f'{key}: {len(df)} rows, expected {EXPECTED[key]}'
        df.to_csv(f'{DERM}/manifest_{key}.csv', index=False)
        print(f'{key}: manifest rebuilt, {len(df)} rows, MEL {int(df.melanoma.sum())}, malignant {int(df.malignant.sum())}')
    json.dump({'rebuilt_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'reason': 'intake manifests did not sync to Drive',
               'counts': {k: len(v) for k, v in MAN.items()}, 'hash_basis': 'processed 512-pixel image'}, open(f'{DERM}/manifest_rebuild_record.json', 'w'), indent=2)
    drive.flush_and_unmount(); drive.mount('/content/drive')
print('manifests present:', all(os.path.exists(f'{DERM}/manifest_{s}.csv') for s in EXPECTED))

manifests present: True


In [4]:
# Cell 2: manifests, cleaning (per the lock), lesion-disjoint splits
M = {s: pd.read_csv(f'{DERM}/manifest_{s}.csv', dtype={'image_id': str, 'lesion_id': str, 'patient_id': str}) for s in ('HAM10000', 'BCN20000', 'HIBA', 'PAD')}
CLEAN = {}
for s, df in M.items():
    n0 = len(df); df = df.sort_values('image_id')
    df['is_scar'] = df.dx_text.astype(str).str.contains('Scar', case=False) & df.dx.isna()
    df['unlabelled'] = df.dx.isna() & ~df.is_scar
    dup = df.duplicated('raw_sha256', keep='first')
    keep = df[~dup & df.dx.isin(CLASSES)].copy(); keep['y'] = keep.dx.map(CI)
    CLEAN[s] = keep
    print(f'{s:9s} {n0} -> {len(keep)} (unlabelled {int(df.unlabelled.sum())}, scar {int(df.is_scar.sum())}, duplicates dropped {int(dup.sum())}) | MEL {int((keep.dx == "MEL").sum())}, malignant {int(keep.malignant.sum())}')
SPLIT_FILE = f'{DERM}/splits_v1.csv'
if os.path.exists(SPLIT_FILE): SPL = pd.read_csv(SPLIT_FILE, dtype={'image_id': str})
else:
    rows = []; rng = np.random.default_rng(20261601)
    for s in ('HAM10000', 'BCN20000'):
        les = np.array(sorted(CLEAN[s].lesion_id.astype(str).unique())); rng.shuffle(les); n = len(les)
        part = {l: ('train' if i < 0.8 * n else 'val' if i < 0.9 * n else 'test') for i, l in enumerate(les)}
        rows += [{'image_id': i, 'source': s, 'split': part[str(l)]} for i, l in zip(CLEAN[s].image_id, CLEAN[s].lesion_id)]
    SPL = pd.DataFrame(rows); SPL.to_csv(SPLIT_FILE, index=False)
for s in ('HAM10000', 'BCN20000'):
    CLEAN[s] = CLEAN[s].merge(SPL[SPL.source == s][['image_id', 'split']], on='image_id')
    print(s, CLEAN[s].split.value_counts().to_dict())
    assert not (set(CLEAN[s][CLEAN[s].split == 'train'].lesion_id) & set(CLEAN[s][CLEAN[s].split != 'train'].lesion_id)), 'lesion leakage'

HAM10000  11720 -> 11720 (unlabelled 0, scar 0, duplicates dropped 0) | MEL 1305, malignant 2156
BCN20000  18946 -> 17476 (unlabelled 1156, scar 314, duplicates dropped 0) | MEL 4636, malignant 8871
HIBA      1616 -> 1616 (unlabelled 0, scar 0, duplicates dropped 0) | MEL 253, malignant 751
PAD       2298 -> 2283 (unlabelled 0, scar 0, duplicates dropped 15) | MEL 52, malignant 1081
HAM10000 {'train': 9359, 'test': 1185, 'val': 1176}
BCN20000 {'train': 14119, 'val': 1750, 'test': 1607}


In [5]:
# Cell 3: copy 512-pixel images to local disk
LOCAL = '/content/derm512'
def _cp(p):
    dst = p.replace(f'{DERM}/images_512', LOCAL)
    if not os.path.exists(dst): os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(p, dst)
    return dst
for s, df in CLEAN.items():
    with ThreadPoolExecutor(16) as ex: df['local'] = list(ex.map(_cp, df.path_512))
    print(s, len(df), 'images on local disk')

HAM10000 11720 images on local disk
BCN20000 17476 images on local disk
HIBA 1616 images on local disk
PAD 2283 images on local disk


In [6]:
# Cell 4: models, data, training loop (DR full recipes)
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from PIL import Image
ARCHS = {'resnet50': ('resnet50', 'cnn'), 'densenet121': ('densenet121', 'cnn'), 'efficientnet_b3': ('efficientnet_b3', 'cnn'),
         'convnext_t': ('convnext_tiny.fb_in1k', 'cnn'), 'vit_b': ('vit_base_patch16_224', 'tfm'), 'vit_b_dinov2': ('vit_base_patch14_dinov2.lvd142m', 'tfm')}
RECIPE = {'cnn': {'epochs': 15, 'lr': 1e-4, 'wd': 1e-4, 'warmup': 0, 'clip': None, 'bs': 32, 'lldecay': None},
          'tfm': {'epochs': 15, 'lr': 3e-5, 'wd': 0.05, 'warmup': 5, 'clip': 1.0, 'bs': 32, 'lldecay': 0.65}}
SOURCES = ['HAM10000', 'BCN20000']; SEEDS = [0, 1]
MODELS = [(s, a, k) for s in SOURCES for a in ARCHS for k in SEEDS]
def mname(s, a, k): return f'{s}_{a}_seed{k}'
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
TF_TRAIN = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                               transforms.ColorJitter(0.2, 0.2, 0.1), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
class DS(Dataset):
    def __init__(self, paths, ys, tf): self.p, self.y, self.tf = list(paths), list(ys), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB')), int(self.y[i]), i
def build(a, pretrained=True):
    tid, rk = ARCHS[a]; kw = {'img_size': 224} if rk == 'tfm' else {}
    return timm.create_model(tid, pretrained=pretrained, num_classes=len(CLASSES), **kw)
def find_blocks(m):
    best = None
    for _, mod in m.named_modules():
        if isinstance(mod, (nn.ModuleList, nn.Sequential)) and len(mod) >= 12 and (best is None or len(mod) > len(best)): best = mod
    return best
def make_optimizer(m, r):
    if r['lldecay'] is None: return torch.optim.AdamW(m.parameters(), lr=r['lr'], weight_decay=r['wd'])
    lr, d = r['lr'], r['lldecay']; blocks = find_blocks(m); n = len(blocks); named = list(m.named_parameters()); bid = {id(p) for b in blocks for p in b.parameters()}
    embed = [p for nm, p in named if id(p) not in bid and nm.startswith(('cls_token', 'pos_embed', 'patch_embed', 'reg_token', 'mask_token'))]
    head = [p for nm, p in named if id(p) not in bid and nm.startswith('head')]
    used = bid | {id(p) for p in embed} | {id(p) for p in head}; rest = [p for nm, p in named if id(p) not in used]
    groups = [{'params': embed, 'lr': lr * d ** n}] + [{'params': list(b.parameters()), 'lr': lr * d ** (n - 1 - i)} for i, b in enumerate(blocks)] + [{'params': rest, 'lr': lr}, {'params': head, 'lr': lr}]
    groups = [g_ for g_ in groups if g_['params']]; assert sum(p.numel() for g_ in groups for p in g_['params']) == sum(p.numel() for p in m.parameters())
    return torch.optim.AdamW(groups, lr=lr, weight_decay=r['wd'])
@torch.no_grad()
def predict(m, paths, bs=128):
    m.eval(); P = np.zeros((len(paths), len(CLASSES)))
    for x, _, idx in DataLoader(DS(paths, [0] * len(paths), TF_EVAL), batch_size=bs, num_workers=NW, pin_memory=True):
        x = x.to(dev)
        with torch.amp.autocast('cuda'): out = m(x).float()
        if not torch.isfinite(out).all(): out = m(x.float()).float()   # fp16 overflow: redo this batch in fp32
        P[idx.numpy()] = torch.softmax(out, 1).double().cpu().numpy()
    return P
def train_one(s, a, k):
    r = RECIPE[ARCHS[a][1]]; seed = 20261601 + 100 * SOURCES.index(s) + 10 * list(ARCHS).index(a) + k
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    D = CLEAN[s]; tr, va = D[D.split == 'train'], D[D.split == 'val']
    m = build(a).to(dev); g = torch.Generator(); g.manual_seed(seed)
    dl_ = DataLoader(DS(tr.local, tr.y, TF_TRAIN), batch_size=r['bs'], shuffle=True, drop_last=True, num_workers=NW, generator=g, pin_memory=True, persistent_workers=True, prefetch_factor=4)
    opt = make_optimizer(m, r); E, W = r['epochs'], r['warmup']
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda ep: (ep + 1) / W if ep < W else 0.5 * (1 + math.cos(math.pi * (ep - W) / max(1, E - W))))
    scaler = torch.amp.GradScaler('cuda'); lossf = nn.CrossEntropyLoss(); best = (-1, None, 0)
    for ep in range(E):
        m.train(); t0 = time.time()
        for x, y, _ in dl_:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'): loss = lossf(m(x), y)
            scaler.scale(loss).backward()
            if r['clip']: scaler.unscale_(opt); nn.utils.clip_grad_norm_(m.parameters(), r['clip'])
            scaler.step(opt); scaler.update()
        sch.step(); Pv = predict(m, va.local); vauc = roc_auc_score(va.malignant.astype(int), Pv[:, MAL].sum(1))
        if vauc > best[0]: best = (vauc, {kk: v.detach().cpu().clone() for kk, v in m.state_dict().items()}, ep + 1)
        print(f'   epoch {ep+1}/{E} val malignant AUC {vauc:.4f} ({time.time()-t0:.0f}s)')
    m.load_state_dict(best[1]); return m, best[0], best[2], seed
for a in ARCHS:   # preflight
    mm = build(a); make_optimizer(mm, RECIPE[ARCHS[a][1]])
    with torch.no_grad(): assert mm(torch.zeros(1, 3, 224, 224)).shape == (1, len(CLASSES))
    del mm
print('preflight ok for', list(ARCHS))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 32.3MB            

model.safetensors: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

preflight ok for ['resnet50', 'densenet121', 'efficientnet_b3', 'convnext_t', 'vit_b', 'vit_b_dinov2']


In [7]:
# Cell 5: train all 24 (resumable)
REC = f'{DERM}/training_record.json'; rec = json.load(open(REC)) if os.path.exists(REC) else {}
for s, a, k in MODELS:
    nm = mname(s, a, k); ck = f'{CK}/{nm}.pt'
    if os.path.exists(ck) and nm in rec: continue
    print(f'=== {nm} ==='); t0 = time.time()
    m, vauc, ep, seed = train_one(s, a, k)
    torch.save({'state_dict': m.state_dict(), 'name': nm, 'source': s, 'arch': a, 'seed_index': k, 'seed': seed, 'val_auc': vauc, 'best_epoch': ep}, ck)
    rec[nm] = {'val_malignant_auc': round(float(vauc), 4), 'best_epoch': ep, 'seed': seed, 'minutes': round((time.time() - t0) / 60, 1)}
    json.dump(rec, open(REC, 'w'), indent=2); print('   done', rec[nm]); del m; torch.cuda.empty_cache()
print(pd.DataFrame(rec).T.to_string())

=== HAM10000_efficientnet_b3_seed0 ===
   epoch 1/15 val malignant AUC 0.8839 (159s)
   epoch 2/15 val malignant AUC 0.9056 (26s)
   epoch 3/15 val malignant AUC 0.9098 (27s)
   epoch 4/15 val malignant AUC 0.9226 (27s)
   epoch 5/15 val malignant AUC 0.9306 (56s)
   epoch 6/15 val malignant AUC 0.9319 (27s)
   epoch 7/15 val malignant AUC 0.9339 (27s)
   epoch 8/15 val malignant AUC 0.9277 (27s)
   epoch 9/15 val malignant AUC 0.9308 (27s)
   epoch 10/15 val malignant AUC 0.9317 (27s)
   epoch 11/15 val malignant AUC 0.9317 (28s)
   epoch 12/15 val malignant AUC 0.9304 (27s)
   epoch 13/15 val malignant AUC 0.9298 (28s)
   epoch 14/15 val malignant AUC 0.9311 (27s)
   epoch 15/15 val malignant AUC 0.9311 (27s)
   done {'val_malignant_auc': 0.9339, 'best_epoch': 7, 'seed': 20261621, 'minutes': 9.5}
=== HAM10000_efficientnet_b3_seed1 ===
   epoch 1/15 val malignant AUC 0.8845 (29s)
   epoch 2/15 val malignant AUC 0.9061 (28s)
   epoch 3/15 val malignant AUC 0.9121 (28s)
   epoch 4/15 va

In [8]:
# Cell 6: GATE, then predictions (external sets only after all 24 checkpoints exist and are hashed)
missing = [mname(*t) for t in MODELS if not os.path.exists(f'{CK}/{mname(*t)}.pt')]
assert not missing, f'not all models trained yet: {missing}'
IREC = f'{DERM}/inference_record.json'
if not os.path.exists(IREC):
    ir = {'hashed_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'lock': 'derm_lock_v1.json',
          'checkpoints': {mname(*t): sha(f'{CK}/{mname(*t)}.pt') for t in MODELS}}
    json.dump(ir, open(IREC, 'w'), indent=2); print('all 24 checkpoints hashed; external sets may now be predicted')
EVAL_SETS = {'HAM10000_test': CLEAN['HAM10000'][CLEAN['HAM10000'].split == 'test'], 'BCN20000_test': CLEAN['BCN20000'][CLEAN['BCN20000'].split == 'test'],
             'HIBA': CLEAN['HIBA'], 'PAD': CLEAN['PAD']}
for s, a, k in MODELS:
    nm = mname(s, a, k); m = None
    for es, D in EVAL_SETS.items():
        out = f'{PR}/{es}__{nm}.csv'
        if os.path.exists(out): continue
        if m is None:
            m = build(a, pretrained=False); m.load_state_dict(torch.load(f'{CK}/{nm}.pt', map_location='cpu', weights_only=False)['state_dict']); m = m.to(dev)
        P = predict(m, D.local)
        df = pd.DataFrame(P, columns=[f'p_{c}' for c in CLASSES]); df.insert(0, 'image_id', D.image_id.values)
        df['p_malignant'] = P[:, MAL].sum(1); df['p_melanoma'] = P[:, CI['MEL']]
        df.to_csv(out, index=False, float_format='%.10g')
    if m is not None: print('predicted', nm); del m; torch.cuda.empty_cache()
print('predictions complete')

all 24 checkpoints hashed; external sets may now be predicted
predicted HAM10000_resnet50_seed0
predicted HAM10000_resnet50_seed1
predicted HAM10000_densenet121_seed0
predicted HAM10000_densenet121_seed1
predicted HAM10000_efficientnet_b3_seed0
predicted HAM10000_efficientnet_b3_seed1
predicted HAM10000_convnext_t_seed0
predicted HAM10000_convnext_t_seed1
predicted HAM10000_vit_b_seed0
predicted HAM10000_vit_b_seed1
predicted HAM10000_vit_b_dinov2_seed0
predicted HAM10000_vit_b_dinov2_seed1
predicted BCN20000_resnet50_seed0
predicted BCN20000_resnet50_seed1
predicted BCN20000_densenet121_seed0
predicted BCN20000_densenet121_seed1
predicted BCN20000_efficientnet_b3_seed0
predicted BCN20000_efficientnet_b3_seed1
predicted BCN20000_convnext_t_seed0
predicted BCN20000_convnext_t_seed1
predicted BCN20000_vit_b_seed0
predicted BCN20000_vit_b_seed1
predicted BCN20000_vit_b_dinov2_seed0
predicted BCN20000_vit_b_dinov2_seed1
predictions complete


In [9]:
# Cell 7: fingerprints for H5 on half A of HIBA patients (unlabelled use only)
import io as _io
from PIL import ImageFilter
H = CLEAN['HIBA']; pats = np.array(sorted(H.patient_id.astype(str).unique())); rng = np.random.default_rng(20261602); rng.shuffle(pats)
HALF_A = set(pats[: len(pats) // 2]); HA = H[H.patient_id.astype(str).isin(HALF_A)]
pd.DataFrame({'patient_id': sorted(HALF_A)}).to_csv(f'{FP}/hiba_half_A_patients.csv', index=False)
BASE224 = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
CLEANIMG = np.stack([np.asarray(BASE224(Image.open(p).convert('RGB')), dtype=np.uint8) for p in HA.local])
def jpeg(a, q): b = _io.BytesIO(); Image.fromarray(a).save(b, 'JPEG', quality=q); return np.asarray(Image.open(b).convert('RGB'))
def blur(a, r): return np.asarray(Image.fromarray(a).filter(ImageFilter.GaussianBlur(r)))
yy, xx = np.mgrid[:224, :224]; VIG = (1 - 0.6 * (((yy - 112) ** 2 + (xx - 112) ** 2) / (2 * 112 ** 2))).clip(0.3, 1)[..., None]
PERTURB = {'lowpass_r2': ('frequency', lambda a: blur(a, 2)), 'lowpass_r4': ('frequency', lambda a: blur(a, 4)),
           'highpass_r4': ('frequency', lambda a: np.clip(a.astype(np.int16) - blur(a, 4).astype(np.int16) + 128, 0, 255).astype(np.uint8)),
           'drop_red': ('channel', lambda a: np.concatenate([np.full_like(a[..., :1], int(a[..., 0].mean())), a[..., 1:]], -1)),
           'drop_green': ('channel', lambda a: np.concatenate([a[..., :1], np.full_like(a[..., 1:2], int(a[..., 1].mean())), a[..., 2:]], -1)),
           'drop_blue': ('channel', lambda a: np.concatenate([a[..., :2], np.full_like(a[..., 2:], int(a[..., 2].mean()))], -1)),
           'jpeg_q15': ('degradation', lambda a: jpeg(a, 15)), 'gamma_dark': ('degradation', lambda a: (255 * (a / 255.0) ** 1.8).astype(np.uint8)),
           'gamma_bright': ('degradation', lambda a: (255 * (a / 255.0) ** 0.55).astype(np.uint8)),
           'noise': ('degradation', lambda a: np.clip(a + np.random.default_rng(0).normal(0, 20, a.shape), 0, 255).astype(np.uint8)),
           'vignette': ('degradation', lambda a: (a * VIG).astype(np.uint8)), 'blur_r3': ('degradation', lambda a: blur(a, 3))}
PERT = {kk: np.stack([f(x) for x in CLEANIMG]) for kk, (_, f) in PERTURB.items()}
MEAN_T = torch.tensor(MEAN).view(1, 3, 1, 1); STD_T = torch.tensor(STD).view(1, 3, 1, 1)
@torch.no_grad()
def run(m, arr, bs=100):
    out = []
    for i in range(0, len(arr), bs):
        x = (((torch.from_numpy(arr[i:i + bs]).permute(0, 3, 1, 2).float() / 255) - MEAN_T) / STD_T).to(dev)
        with torch.amp.autocast('cuda'): o = m(x).float()
        if not torch.isfinite(o).all(): o = m(x).float()   # fp16 overflow: redo this batch in fp32
        out.append(torch.softmax(o, 1)[:, MAL].sum(1).double().cpu())
    return torch.cat(out).numpy()
for s, a, k in MODELS:
    nm = mname(s, a, k); fp = f'{FP}/{nm}.npz'
    if os.path.exists(fp): continue
    m = build(a, pretrained=False); m.load_state_dict(torch.load(f'{CK}/{nm}.pt', map_location='cpu', weights_only=False)['state_dict']); m = m.to(dev).eval()
    base = run(m, CLEANIMG); resp = {f'resp_{kk}': run(m, PERT[kk]) - base for kk in PERTURB}
    np.savez_compressed(fp, image_id=HA.image_id.values, clean=base, **resp); print('fingerprint', nm); del m; torch.cuda.empty_cache()
print('Step 2 complete: 24 models, predictions on 4 sets, fingerprints on HIBA half A. Send training_record.json and inference_record.json.')

fingerprint HAM10000_resnet50_seed0
fingerprint HAM10000_resnet50_seed1
fingerprint HAM10000_densenet121_seed0
fingerprint HAM10000_densenet121_seed1
fingerprint HAM10000_efficientnet_b3_seed0
fingerprint HAM10000_efficientnet_b3_seed1
fingerprint HAM10000_convnext_t_seed0
fingerprint HAM10000_convnext_t_seed1
fingerprint HAM10000_vit_b_seed0
fingerprint HAM10000_vit_b_seed1
fingerprint HAM10000_vit_b_dinov2_seed0
fingerprint HAM10000_vit_b_dinov2_seed1
fingerprint BCN20000_resnet50_seed0
fingerprint BCN20000_resnet50_seed1
fingerprint BCN20000_densenet121_seed0
fingerprint BCN20000_densenet121_seed1
fingerprint BCN20000_efficientnet_b3_seed0
fingerprint BCN20000_efficientnet_b3_seed1
fingerprint BCN20000_convnext_t_seed0
fingerprint BCN20000_convnext_t_seed1
fingerprint BCN20000_vit_b_seed0
fingerprint BCN20000_vit_b_seed1
fingerprint BCN20000_vit_b_dinov2_seed0
fingerprint BCN20000_vit_b_dinov2_seed1
Step 2 complete: 24 models, predictions on 4 sets, fingerprints on HIBA half A. Send